## In this session

- **Background:** in the Dense assignment you trained nets of different depths and widths on ship images and saw that results can be erratic
- **This session:** an overview of the small decisions that decide whether a network trains (activation, initial weights, centering, scale with batch normalization, and size), the one thread behind most of them, and a table of symptoms
- **By the end you can**
    - name the small decisions and what goes wrong with each
    - say what "where the pre-activations land" means
    - read a symptom (a flat loss, dead units, huge gradients) and say what to check first

## Small decisions, big effects: the overview

| Decision | What can go wrong | One-line intuition | Usual fix | Detail |
|---|---|---|---|---|
| **Activation function** | sigmoid: the **gradient** (the signal that tells each weight how to change) shrinks layer by layer | each sigmoid layer multiplies the gradient by at most 0.25, so it shrinks fast with depth; tanh and ReLU have slopes up to 1, so the slopes alone do not force shrinking | tanh or ReLU in hidden layers (sigmoid mainly at the output), with $z$ kept in a good range (next slides) | [Vanishing and exploding gradients](Vanishing_and_Exploding_Gradients.ipynb) |
| **Initial weights** | units that start identical stay identical; all zeros: no gradient reaches the hidden layers at the first step | nothing breaks the tie between units | small random weights, different for every unit (**Glorot / He**: random with a spread set by the layer's size) | [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb) |
| **Centering** | all-positive inputs make a unit's weight updates share one sign (**zig-zag**: the weights lurch back and forth instead of heading straight for the minimum); sigmoid outputs average 0.5, so every layer is uncentered | all the weights get pushed the same way at once | center the inputs; prefer tanh to sigmoid | [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb) |
| **Scale** | too small: nearly linear, tiny signal; too large: **saturated** (stuck in a flat part of the curve; sigmoid, tanh) or blown up (ReLU) | keep layer inputs at a moderate scale (roughly unit variance, not exactly) | Glorot / He initialization; **batch normalization**: a new layer type that re-centers and rescales a layer's pre-activations $z$ so the next layer gets better-behaved inputs (it then learns its own center and scale) | [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb) |
| **Network size** | a net with too few units per layer can fail to train | a layer with 2 to 3 units often starts dead (ReLU) or trains slowly (tanh) | enough units per layer; **prune** (delete the least useful weights) afterwards only to compress a big net | [How big should my NN be?](Training_Neural_Networks_Network_Size.ipynb) |

- The activation and scale rows act through one thread: **where the pre-activations land** (next slides)
- Initial weights matter twice: their scale moves where $z$ lands (see Scale), and they must differ from unit to unit (see Initial weights)
- Centering (the shape of the loss surface) and network size are separate matters; their slides say so

## The thread: where do the pre-activations land?

A **unit** is one neuron. Its **pre-activation** is $z = \mathbf{w}\cdot\mathbf{x} + b$ (weights times inputs, plus bias), the input to its activation function $a(z)$.

<img src="images/preact_thread.png" width=80%>

- initialization (through its scale) and scale or normalization move **where $z$ lands**; the choice of activation decides **which region of $a(z)$ is favorable**; centering matters mostly through the loss surface (slide on centering), and a large shift also pushes $z$ toward saturation
- training needs two things from each unit
    - **slope** $a'(z)$ (the derivative of $a$) not near $0$: the gradient gets through and the weights into the unit update
    - **non-linearity** (curvature or a kink): otherwise a stack of layers collapses to one linear map (a composition of linear maps is linear)
- two effects are **not** about where $z$ lands, and their slides say so: identical starting weights (a symmetry problem) and uncentered inputs (the shape of the loss surface)
- "favorable" is a **region**, not a point. Next: how the slope compounds across layers (activation choice), then what the favorable region looks like for sigmoid/tanh (we vary the spread of $z$) and for ReLU (we vary its position)

## Activation choice: sigmoid's small slope shrinks the gradient layer by layer

<img src="images/preact_vanishing.png" width=90%>

- going backward through a layer, the gradient is multiplied by the activation's slope $a'(z)$ and by the layer's weights (chain rule, schematically: $\delta_{\text{layer}} = a'(z) \odot (W^\top \delta_{\text{next layer}})$, where $\odot$ multiplies element by element)
- **sigmoid**: $a'(z) \le 0.25$ always, so the slope factors alone cut the gradient to at most $0.25^k$ after $k$ layers
    - 10 layers: at most about $10^{-6}$ of the original size; 20 layers: about $10^{-12}$
    - the early layers then barely learn: **vanishing gradients**
- **tanh and ReLU**: the slope can reach $1$, so the *best case* is no shrinking at all, but the actual product can be anywhere between $1$ and $0$
    - an adverse $z$ makes it small (the next two slides show where $z$ lands)
    - a saturated tanh unit has slope near $0$; even at moderate spread tanh's *average* slope is well below $1$ (e.g. $0.36$ at $s = 2$)
    - a ReLU unit with $z < 0$ has slope exactly $0$ and passes nothing
- the *right* panel shows the **best case** (an upper bound), not a measurement, and it uses the slopes only; the weights also multiply the gradient (they can make it explode instead)
- Detail: [Vanishing and exploding gradients](Vanishing_and_Exploding_Gradients.ipynb)

## Sigmoid and tanh: what happens as the spread of $z$ changes

<img src="images/preact_sigmoid_tanh.png" width=62%>

Grey bars: the share of the $z$ values (in %) that fall in each bin of width $0.2$, i.e. where the pre-activations actually land. Note the vertical scales differ from column to column.

Numbers under each panel: **avg slope** = the average of $a'(z)$ over the $z$ values (how much gradient gets through); **non-linearity** = the share of the variation of $a(z)$ that a straight line in $z$ cannot explain ($0\%$ = a straight line).

- the curves are the same in every column; only **where $z$ lands** (grey) changes
- **narrow $z$** (small $s$): the slope is near its maximum ($0.25$ for sigmoid, $1$ for tanh), but the unit is almost a straight line
- **wide $z$** (large $s$): clearly non-linear, but most $z$ sit in the flat tails where the slope is about $0$
- the trade-off is the point: as the spread $s$ grows, the slope falls and the non-linearity rises
- sigmoid never gets both (its slope is at most $0.25$); tanh is better balanced at moderate $s$
- *Synthetic illustration:* $z \sim \mathcal{N}(0, s^2)$, so $s$ is the standard deviation of $z$ across examples (it is not a property of the slope). The averages include the tails beyond $\pm 6$ that are not drawn (about a quarter of the $z$ at $s = 5$)
- Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## ReLU: what matters is where $z$ sits relative to $0$

<img src="images/preact_relu.png" width=90%>

- $z < 0$: slope $0$, no gradient. $z > 0$: slope $1$, a straight line
- the non-linearity is the **kink**: a layer is non-linear because different units and examples fall on different sides of $0$
- here the spread of $z$ is fixed and its **position** varies
- **mostly negative** (e.g. a very negative bias): the unit is nearly dead; a unit whose $z$ is negative for *every* example gets no update to its own weights, so it does not recover on its own
- **mostly positive**: the unit is nearly linear and the kink is unused
- what keeps $z$ on both sides of $0$: the scale and batch-norm slides
- for $z$ centered at $0$, about half of a unit's examples activate it whatever the spread: for the share of active units the *position* matters, not the spread (the *size* of the outputs still grows with the spread: see the scale slide)
- *Synthetic illustration:* $z \sim \mathcal{N}(\mu, 1)$ with $\mu = -2, 0, 2$. Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## Initial weights: units that start the same stay the same

**Setup:** a Dense layer of **4 hidden units** (tanh), fed 2 inputs and followed by 1 output unit, trained by gradient descent. Each curve is one hidden unit's weight on input 1; weights change **only through gradient updates**, so any slow drift is the optimizer at work, not the activation.

<img src="images/preact_identical.png" width=90%>

- if every hidden unit starts with the same weights (and the weights after them are the same too), each computes the same output from the same inputs, so each receives the same gradient and the same update
    - they stay copies forever: a layer of 4 units behaves like **one** unit
- random, different starting weights break the tie, so units can become different features
- an **all-zero** start is the extreme case: no gradient reaches the hidden layers at the first step, and with tanh the output weights get none either, so the loss stays at $\ln 2$ (and the units are identical anyway)
- nobody initializes identically on purpose; this is why initialization must be *random*
- *Synthetic illustration (one run each, so an anecdote, not a statistic):*
    - 1 hidden layer of 4 tanh units, two concentric rings of 2-D points (chance accuracy is $50\%$), 400 gradient-descent steps
    - left: every weight starts at $0.5$; final accuracy $60\%$. Right: random starts; $99\%$
    - the plot shows one weight per unit; all of an identical unit's weights stay identical
- Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## Centering: all-positive inputs make training zig-zag

<img src="images/preact_centering.png" width=85%>

- when inputs are large and all positive (here mean 5, spread 1), the loss surface is a **long narrow valley** (the condition number is the ratio of its steepest to its flattest curvature)
    - why: a weight's update is (the unit's error signal) $\times$ (its input); with all inputs positive, all the weights of a unit are pushed the same way at once
    - so gradient descent bounces across the valley and crawls along it
- centering the inputs makes the surface rounder and the path direct
- the same issue repeats inside the network: **sigmoid** outputs are always positive (average about $0.5$ for $z$ centered at $0$), so every sigmoid layer hands uncentered inputs to the next one
    - tanh outputs lie in $(-1, 1)$ and average near $0$ for symmetric $z$: one reason it is preferred to sigmoid
- the gap shows up in the **step size**: $0.036$ (uncentered) and $0.87$ (centered), about $25\times$ apart
    - even the small step bounces and crawls
    - on the round surface a large step lands next to the optimum in one move (so the path has a single visible segment)
- *Synthetic illustration:* a 2-weight linear model, inputs with mean 5 vs. mean 0, 60 steps, each panel with a step size near the largest stable one for its surface
- Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## Scale: keep the spread of $z$ moderate through every layer

<img src="images/preact_scale.png" width=85%>

- in a Dense layer each unit's pre-activation is a **sum of $n$ terms**: $z = \sum_{j=1}^{n} w_j x_j + b$ ($n$ = number of inputs to the layer)
- if the inputs $x_j$ and the weights $w_j$ are independent with mean $0$, the inputs share one variance $\text{Var}(x)$ and the weights share one variance $\sigma_w^2$, then $\text{Var}(z) \approx n\,\sigma_w^2\,\text{Var}(x)$
    - so each layer multiplies the variance by about $n\,\sigma_w^2$ (the spread by $\sqrt{n}\,\sigma_w$); with $\sigma_w^2 = 1$ the variance grows $n$-fold
    - **too small** ($n\,\sigma_w^2 \ll 1$): $z$ shrinks toward $0$: nearly linear units, tiny signal and gradients
    - **too large** ($n\,\sigma_w^2 \gg 1$): tanh units saturate at $\pm 1$ and $z$ stays large (flat orange line: tanh bounds its outputs); a ReLU net has no such bound and blows up instead
- **Glorot / He initialization** sets $\sigma_w$ from the layer sizes ($\sigma_w^2 \approx 1/n$ for tanh (LeCun; Glorot's $2/(n_{in}+n_{out})$ equals it when both are $n$), $2/n$ for ReLU (He)) so that $n\,\sigma_w^2 \approx 1$ at the start (He's $2/n$ compensates for ReLU zeroing half the units; Glorot uses both the number of inputs and of outputs)
    - green line: about right, though it still sinks slowly with depth because tanh outputs are smaller than $z$
    - once training changes the weights, the spread can drift away from where initialization put it (next slide: batch normalization)
- "about right" is a **compromise**, not a bullseye: a spread near $0.3$ keeps the gradient large but the unit nearly linear (previous slides), while a spread of $1$ to $2$ uses the non-linearity; unit variance is a rule of thumb
- *Synthetic illustration:* 8 tanh layers, width 256, Gaussian inputs and random weights, no training
- Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## Batch normalization: re-impose a good scale at every layer

<img src="images/preact_bn.png" width=85%>

- **batch normalization**: a new layer type that re-centers and rescales a layer's pre-activations $z$ (using the current mini-batch) so the next layer gets better-behaved inputs
- it does this on every forward pass, so right after it the spread of $z$ no longer depends on how the weights were initialized or how training has changed them (the learned scale then sets it)
- then it lets the network **learn its own center and scale** (two learnable values per unit), so it is not forced to keep $z$ exactly at mean $0$, variance $1$
- left: the three nets from the previous slide; right: the same nets with batch norm (learnable center and scale still at their starting values $0$ and $1$)
- the spread is $1$ at every layer, **by construction**: this shows what batch norm does to the spread, not that training improves (the next slide shows training)
- *Synthetic illustration:* same 8 tanh layers, width 256, no training
- Detail: [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb)

## Do these decisions show up in training?

So far each decision was shown in isolation. Here the same small classifier is trained many times, changing **mainly one decision at a time**, to see how much each one matters.

**Setup:** two concentric rings of 2-D points; 4 hidden layers of 5 units; cross-entropy loss ($0.69 = \ln 2$ is chance level for two balanced classes); Adam (a gradient-descent variant) with learning rate $0.01$ on all 1000 points at every step (full batch); 200 epochs (full passes over the data); 10 random seeds (random starting weights). In each panel the line is the median over seeds and the band runs from the best to the worst seed. "Success" means $90\%$ **training** accuracy at the end. **Baseline:** tanh with Glorot initialization and centered inputs (sigmoid in the batch-norm panel); each panel changes mainly one thing from it (not strictly: ReLU also uses He initialization, and sigmoid outputs are uncentered).

<img src="images/exp_curves.png" width=62%>

- **Activation:** tanh and ReLU succeed in 10/10 runs; sigmoid in 4/10 (median final loss $0.60$, where $0.69$ is chance level)
- **Initial weights:** random (Glorot) 10/10; all weights $0.5$ or all zeros 0/10
- **Input centering:** shifting the inputs by $+3$ drops tanh from 10/10 to 4/10 (sigmoid: 4/10 to 0/10)
- **Batch norm:** sigmoid goes from 4/10 to 10/10 (why was not isolated: it also re-centers the inputs of every layer)
- **All-0.5 and all-zero starts:** identical starts give identical runs, so "0/10" is really one run repeated
- *Caveats:* one small problem, one optimizer and learning rate, 200 epochs; some "failures" might finish with more epochs (e.g. the tanh run with shifted inputs is still improving, median loss $0.3$)
- these runs show that each decision **matters**, not **why**: the mechanism slides give the likely cause (e.g. a shift of $+3$ also pushes the first layer's $z$ toward saturation, and Adam is less sensitive to conditioning than plain gradient descent)

## How do I see it? Look inside the network

The slopes and spreads of the earlier slides are invisible while training, but they can be measured. The left panel measures how large the gradient is in each layer of a 10-layer network at initialization; the right panel shows what too-large weights do, on the same vertical scale. (Same two-ring data; median over 10 seeds, 3 seeds for the exploding case.)

<img src="images/exp_grad.png" width=80%>

- **Gradient size per layer** (at initialization; 10 layers of 20 units):
    - sigmoid: the gradient at layer 1 is about $3 \times 10^{-8}$ of the one at layer 10 (medians $1.8 \times 10^{-10}$ vs. $6.1 \times 10^{-3}$): **vanishing**
    - tanh and ReLU (He): about the same size at every layer
- **Too-large weights** (ReLU, weights $\mathcal{N}(0,1)$, 256 units): gradients around $10^{7}$ to $10^{8}$ at every layer, on the *same* vertical scale: **exploding**
    - this comes from the weights, not the activation: ReLU does not protect against it
- What to log while training:
    - the gradient size of each layer
    - a histogram of $z$ per layer (the earlier slides: too narrow, too wide, mostly negative)
    - the fraction of ReLU units that are never active
    - the loss: staying near $0.69$ ($\ln 2$) in a balanced binary problem means nothing has been learned yet (the sigmoid runs above sit there for about 140 epochs before some escape)

## How big should the network be?

The last small decision is how many units each layer has. We repeat the running example with the **number of units per hidden layer** varied (4 hidden layers, 20 seeds each; ReLU with He initialization and tanh with Glorot initialization; success means $90\%$ accuracy). Left: how often training succeeds. Right: how many ReLU units end up never active.

<img src="images/exp_width.png" width=75%>

- **too narrow trains poorly:** with 2 units per layer ReLU succeeds in 0/20 runs and tanh in 1/20; with 3 units, ReLU 10/20 and tanh 17/20
- **5 units or more trains reliably** (ReLU 20/20 at 5 and 10 units, 19/20 at 50; tanh 20/20)
- the failures at 2 to 3 units are mostly a **training** problem, not too little capacity: ReLU runs that end at chance all have a whole layer of dead units, and tanh runs mostly just train slowly (longer training rescues many; detail notebook)
- the surprise: ReLU nets with 5 or more units train fine even though about $10\%$ to $19\%$ of their units end up **dead** (never active; $39\%$ at 2 units): nets with 5 or more units tolerate some dead units, and the dead fraction is not monotone in width
- practical rule: give each layer enough units (here about 5 or more)
- the **Lottery Ticket** idea: a big randomly initialized network contains small sub-networks that train well; one can train big and then **prune** (delete the least useful weights) to compress it. The detail notebook tests it on a toy: sparse nets can be trained, but the original initial weights did not matter there
- Detail: [How big should my NN be?](Training_Neural_Networks_Network_Size.ipynb)

## What we established, and what it sets up

- training a network is sensitive to a few **small decisions**: activation, initial weights, centering, scale, and the size of the network
- the activation and scale decisions act through one thread: **where the pre-activations $z$ land** on the activation function; identical weights (symmetry), centering (loss surface) and size are separate effects
    - slope near $0$: the gradient does not get through, so the weights into the unit do not update
    - no curvature or kink in use: the unit is just linear
- a quick way to read the symptoms; **check first** the loss (stuck near $0.69$?), then the gradient size per layer, then the share of dead units, then initialization and scale

| If you see | Suspect | Usual fix | Slide |
|---|---|---|---|
| loss stuck near $0.69$ from the start | nothing is learned: a dead layer, identical or all-zero weights, or vanished gradients | look inside the net (gradient per layer, dead units); fix the initialization | How do I see it? |
| early layers barely learn (sigmoid) | gradient shrinking layer by layer | tanh or ReLU, with $z$ kept in a good range | Activation choice |
| a unit never activates (ReLU) | $z$ negative for every example | keep $z$ spread around $0$ (init, normalization); ReLU variants with a non-zero slope (e.g. leaky ReLU) are in [Vanishing and exploding gradients](Vanishing_and_Exploding_Gradients.ipynb) | ReLU: where $z$ sits |
| a layer of units behaving like one unit | identical starting weights | random, different weights | Initial weights |
| training bounces and crawls | uncentered inputs or layer outputs (a too-large step size, the learning rate, looks similar) | center inputs; tanh over sigmoid | Centering |
| signal shrinking or saturating with depth | weight scale too small or too large | Glorot / He; batch normalization | Scale; Batch normalization |
| gradients are enormous (the loss may blow up) | weights too large (not an activation problem) | Glorot / He initialization; gradient clipping (cap the gradient's size) | How do I see it? |
| a small net fails where a bigger one trains | too few units (ReLU: a whole layer dies; tanh: slow training) | enough units per layer (about 5 or more in this toy problem) | How big should the network be? |

- **Also matters, but a different thread:** learning-rate schedules (optimization) and regularization (generalization: dropout, weight penalties, label smoothing, mixup): [Learning-rate schedules and regularization](Training_Neural_Networks_Tweaks.ipynb)
- Details for each cause: [Vanishing and exploding gradients](Vanishing_and_Exploding_Gradients.ipynb), [Scaling and initialization](Training_Neural_Networks_Scaling_and_Initialization.ipynb), [How big should my NN be?](Training_Neural_Networks_Network_Size.ipynb)